# Basic Testing

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from emergence.data.tokenizer import CharacterTokenizer
from emergence.data.datasets import NextTokenDataset
from emergence.models.transformer.config import TransformerConfig
from emergence.models.transformer.model import TransformerModel
from emergence.training.train import train_model

In [ ]:
PROJECT_ROOT = Path.cwd()

DATASET_PATH = PROJECT_ROOT / "data" / "shakespeare.txt"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_PATH)
print("Checkpoints:", CHECKPOINT_DIR)
print("Outputs:", OUTPUT_DIR)

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"

In [ ]:
text = DATASET_PATH.read_text(encoding="utf-8")

tokenizer = CharacterTokenizer()
tokenizer.fit(text)

token_ids = tokenizer.encode(text)

print("Dataset characters:", len(text))
print("Vocabulary size:", tokenizer.vocab_size)

CONTEXT_LENGTH = 256

dataset = NextTokenDataset(
    token_ids=token_ids,
    context_length=CONTEXT_LENGTH,
)

train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)

print("Training samples:", len(dataset))

In [ ]:
config = TransformerConfig(
    vocab_size=tokenizer.vocab_size,
    context_length=CONTEXT_LENGTH,
    embedding_dim=128,
    num_layers=4,
    num_heads=4,
    dropout=0.1,
)

model = TransformerModel(config)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model.to(device)

print("Device:", device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
run = train_model(
    model=model,
    train_loader=train_loader,
    epochs=15,
    learning_rate=3e-4,
    checkpoint_dir=CHECKPOINT_DIR,
)

print("Training complete!")
print("Best loss:", run.best_loss)
print("Completed epochs:", run.completed_epochs)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(run.history) + 1), run.history)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()

In [ ]:
import json

results = {
    "epochs": run.completed_epochs,
    "best_loss": run.best_loss,
    "history": run.history,
    "config": {
        "context_length": CONTEXT_LENGTH,
        "embedding_dim": 128,
        "num_layers": 4,
        "num_heads": 4,
        "dropout": 0.1,
        "learning_rate": 3e-4,
        "batch_size": 32,
    },
}

(OUTPUT_DIR / "results.json").write_text(
    json.dumps(results, indent=4),
    encoding="utf-8",
)

In [ ]:
from emergence.generation.generator import TextGenerator
from emergence.data.tokenizer import CharacterTokenizer

tokenizer = CharacterTokenizer()
tokenizer.fit(DATASET_PATH.read_text(encoding="utf-8"))

generator = TextGenerator.from_checkpoint(
    checkpoint_path=CHECKPOINT_DIR / "best.pt",
    config=config,
    tokenizer=tokenizer,
    device=device,
)
while True:
    prompt = input("Prompt (or 'exit'): ")

    if prompt.strip().lower() == "exit":
        break

    if not prompt:
        continue

    generated_text = generator.generate(
        prompt=prompt,
        max_new_tokens=300,
        temperature=0.8,
    )

    print("\n" + generated_text)
    print("\n" + "-" * 60 + "\n")